# Convert Tim's BrainSpan MutPred2 file to the `MutPred2_gtex.tsv` format

The IsoPath step (`exTSP_compute/mutPred2IsoPath.py`) reads `config.MutPred2_brainspan_file`
(`data/filtered/MutPred2_brainspan.tsv`) and expects the same four columns as `MutPred2_gtex.tsv`:

| GTEx format | built from Tim's column(s) |
|---|---|
| `Variant` | `#CHROM`, `POS`, `REF`, `ALT` → `chr_pos_REF/ALT` |
| `ENST` | `transcript` |
| `aa_sub` | `Substitution` |
| `MutPred2` | `MutPred2 score` |

Tim's `protein` (ENSP) column is dropped because the GTEx file has no equivalent.

**Keeping the original.** Tim's raw file had been moved to `filtered/MutPred2_brainspan.tsv`, which is
the path this notebook writes to. Step 2 therefore first copies the raw file back to
`MutPred2_scores/MutPred2_scores_ASD_brainspan.tsv`, and the conversion always reads from that copy.
Re-running the notebook is safe.

## 1. Setup

In [1]:
import shutil
from pathlib import Path

import pandas as pd

from ExTSP.config import DATA_DIR, MutPred2_gtex_file, MutPred2_brainspan_file

DATA     = Path(str(DATA_DIR))
RAW_TIM  = DATA / "MutPred2_scores" / "MutPred2_scores_ASD_brainspan.tsv"   # Tim's original, kept untouched
OUT      = Path(str(MutPred2_brainspan_file))                               # what the pipeline reads
GTEX     = Path(str(MutPred2_gtex_file))                                    # format reference

RAW_COLUMNS = ["#CHROM", "POS", "REF", "ALT", "protein", "transcript", "Substitution", "MutPred2 score"]

## 2. Preserve Tim's raw file

If the raw copy doesn't exist yet and the output path still holds Tim's raw format, copy it across
before anything is overwritten.

In [2]:
if not RAW_TIM.exists():
    header = pd.read_csv(OUT, sep="\t", nrows=0).columns.tolist()
    assert header == RAW_COLUMNS, f"{OUT} is not Tim's raw file (columns {header}); refusing to guess"
    shutil.copy2(OUT, RAW_TIM)
    print(f"copied raw file\n  {OUT}\n-> {RAW_TIM}")
else:
    print(f"raw copy already present: {RAW_TIM}")

raw copy already present: /workspaces/ExTSP-default-claude/src/ExTSP/data/MutPred2_scores/MutPred2_scores_ASD_brainspan.tsv


## 3. Load Tim's file and the GTEx reference

In [3]:
raw  = pd.read_csv(RAW_TIM, sep="\t")
gtex = pd.read_csv(GTEX, sep="\t", nrows=5)
assert raw.columns.tolist() == RAW_COLUMNS

print(f"Tim's file   : {len(raw)} rows, columns {raw.columns.tolist()}")
print(f"GTEx format  : columns {gtex.columns.tolist()}")
gtex

Tim's file   : 1738 rows, columns ['#CHROM', 'POS', 'REF', 'ALT', 'protein', 'transcript', 'Substitution', 'MutPred2 score']
GTEx format  : columns ['Variant', 'ENST', 'aa_sub', 'MutPred2']


,Variant,ENST,aa_sub,MutPred2
0,10_100209695_C/T,ENST00000370397.7,R343H,0.080
1,10_100293374_C/A,ENST00000318222.3,M555I,0.822
2,10_100297026_C/T,ENST00000318222.3,R380H,0.053
3,10_100529393_C/T,ENST00000299166.8,D67N,0.531
4,10_100529393_C/T,ENST00000370320.4,D67N,0.170


## 4. Convert

Rows with a blank MutPred2 score are dropped. `MutPred2_gtex.tsv` contains none, and the IsoPath step
would drop them anyway.

Tim's file also repeats some rows exactly: same variant, isoform, substitution and score. These are
collapsed to one row. If the same (variant, isoform, substitution) ever appeared with *different* scores,
the notebook stops instead of picking one. Rows are sorted by variant then isoform, as in the GTEx file.

In [4]:
bs = pd.DataFrame({
    "Variant":  raw["#CHROM"].astype(str) + "_" + raw["POS"].astype(str) + "_" + raw["REF"] + "/" + raw["ALT"],
    "ENST":     raw["transcript"],
    "aa_sub":   raw["Substitution"],
    "MutPred2": raw["MutPred2 score"],
})

blank = bs["MutPred2"].isna()
print(f"rows with a blank score (dropped): {blank.sum()}  "
      f"({bs.loc[blank, 'Variant'].nunique()} variants; "
      f"{len(set(bs.loc[blank, 'Variant']) - set(bs.loc[~blank, 'Variant']))} of them have no score on any isoform)")
bs = bs[~blank]

n = len(bs)
bs = bs.drop_duplicates()
print(f"exact repeated rows collapsed: {n - len(bs)}")
bs = bs.sort_values(["Variant", "ENST"]).reset_index(drop=True)

assert not bs.duplicated(["Variant", "ENST", "aa_sub"]).any(), "same (variant, isoform, substitution) with conflicting scores"
assert bs.columns.tolist() == gtex.columns.tolist()
assert bs["MutPred2"].between(0, 1).all()

print(f"converted: {len(bs)} rows, {bs['Variant'].nunique()} variants, {bs['ENST'].nunique()} isoforms")
bs.head()

rows with a blank score (dropped): 104  (30 variants; 30 of them have no score on any isoform)
exact repeated rows collapsed: 36
converted: 1598 rows, 448 variants, 542 isoforms


,Variant,ENST,aa_sub,MutPred2
0,10_113152414_C/G,ENST00000277945.8,P132A,0.306
1,10_113152414_C/G,ENST00000352065.6,P392A,0.695
2,10_113152414_C/G,ENST00000355717.5,P439A,0.601
3,10_113152414_C/G,ENST00000355995.5,P415A,0.494
4,10_113152414_C/G,ENST00000369386.2,P58A,0.332


## 5. Write and read back

In [5]:
bs.to_csv(OUT, sep="\t", index=False)

check = pd.read_csv(OUT, sep="\t")
assert check.columns.tolist() == gtex.columns.tolist()
assert check.dtypes.to_dict() == pd.read_csv(GTEX, sep="\t", nrows=1000).dtypes.to_dict()
assert len(check) == len(bs)
print(f"wrote {len(check)} rows -> {OUT}")
print("columns and dtypes match MutPred2_gtex.tsv")

wrote 1598 rows -> /workspaces/ExTSP-default-claude/src/ExTSP/data/filtered/MutPred2_brainspan.tsv
columns and dtypes match MutPred2_gtex.tsv
